# 01 — Dataset — Classification par graveur (cellule a mettre a jour)

**Projet** : Gallica Images — Illustrations des *Métamorphoses* d'Ovide  
**Date**   : Avril 2026

Ce notebook construit le dataset pour entraîner un classifieur par graveur.
Les illustrations déjà segmentées dans `data/segmentees/` sont réutilisées directement.
Les nouvelles sources ont été téléchargées depuis l'API BnF, BSB Munich ou extraites depuis des PDFs.

## Graveurs et sources

| Graveur | Dossier(s) segmenté(s) | Nb | Statut |
|---|---|---|---|
| Salomon, Bernard | `bois_salomon_lyon1557` | 164 | ✓ API BnF |
| Solis, Virgil | `bois_solis_francfort1581` | 187 | ✓ BSB Munich IIIF |
| Wickram, Jörg | `bois_wickram_mayence1545` | 52 | ✓ BSB Munich IIIF |
| Savery, Salomon | `cuivre_clein_paris1637` | 18 | ✓ BSB Munich IIIF |
| Passe, Crispin de | `cuivre_depasse_koln1602` + `cuivre_depasse_arnhem1607` | 271 | ✓ PDF Gallica |
| Eskrich, Pierre | `bois_eskrich_rouille1556` | 56 | ✓ API BnF |
| Leroy II, Guillaume | `bois_leroy_gueynard1510` | 44 | ✓ BSB Munich IIIF |
| Gaultier, Léonard | `cuivre_gaultier_guillemot1610` + `cuivre_gaultier_1616` + `cuivre_gaultier_veuveguillemot1614` | 74 | ✓ BSB Munich IIIF + PDF Gallica |
| Isaac, Jaspar | `cuivre_isaac_langelier1617` | 17 | ✓ PDF Gallica |
| Briot, Isaac | `cuivre_briot_drobet1628` | 49 | ✓ PDF téléchargé |
| Goltzius, Hendrick | `cuivre_goltzius_haarlem1589` | 38 | ✓ PDF téléchargé — converti en niveaux de gris |
| Baur, Johann Wilhelm | — | 0 | ✗ exemplaire physique Bnu Strasbourg |
| Altzenbach, Gerhardt | — | 0 | ✗ exemplaire physique Bnu Strasbourg |
| Mulder, Joseph | — | 0 | ✗ à sourcer |
| Weyen, Laurent | — | 0 | ✗ à sourcer |
| Blanchin, Jean | — | 0 | ✗ à sourcer |
| Philippe, Pierre | — | 0 | ✗ entrée manquante dans le tableau |

**Total actuel : 970 illustrations — 11 graveurs**

---

**Sorties :**
- `data/segmentees/{source}/` — illustrations segmentées par source
- `data/datasets/graveur/{graveur}/` — illustrations rassemblées par graveur
- `data/datasets/graveur/train/val/test/` — split final

---

## 1. Configuration

In [4]:
import sys
sys.path.insert(0, "../../")  # remonte à notebooks/
from gallica_utils import charger_yolo, segmenter_corpus, liberer_yolo, telecharger_pages_iiif

import os, shutil, random, re
import torch
from PIL import Image, ImageOps

RACINE          = os.path.abspath("../../../")   # working_dir
DOSSIER_SEG     = os.path.join(RACINE, "data", "segmentees")
DOSSIER_SOURCES = os.path.join(RACINE, "data", "sources")
DOSSIER_DATASET = os.path.join(RACINE, "data", "datasets", "graveur")
YOLOV5_REPO     = os.path.join(RACINE, "yolov5_repo")

os.makedirs(DOSSIER_DATASET, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Racine  : {RACINE}")
print(f"Device  : {device}")

Racine  : /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir
Device  : cuda


## 2. Correspondance graveur → dossiers segmentés

Mapping entre le nom du graveur et les dossiers déjà segmentés.

In [23]:
# Mapping graveur → liste de dossiers segmentés
# Clé   : nom du graveur (sera le nom de la classe)
# Valeur : liste de dossiers dans data/segmentees/
# Pour cuivre_pdfs les dossiers sont dans un sous-dossier

GRAVEURS = {
    "salomon"  : ["bois_salomon_rouille_lyon1557"],
    "solis"    : ["bois_solis_feyerabend_francfort1581"],
    "wickram"  : ["bois_wickram_behem_mayence1545"],
    "savery"   : ["cuivre_savery_farnaby_paris1637"],
    "de_passe" : ["cuivre_depasse_depasse_koln1602",
                  "cuivre_depasse_jansonius_arnhem1607"],
    "eskrich"  : ["bois_eskrich_rouille_lyon1556"],
    "leroy"    : ["bois_leroy_gueynard_lyon1510"],
    "gaultier" : ["cuivre_gaultier_guillemot_paris1610",
                  "cuivre_gaultier_sn_paris1616",
                  "cuivre_gaultier_veuveguillemot_paris1614"],
    "isaac"    : ["cuivre_isaac_langelier_paris1617"],
    "briot"    : ["cuivre_briot_drobet_lyon1628"],
    "goltzius" : ["cuivre_goltzius_goltzius_haarlem1589"],

    "blanchin" : ["cuivre_blanchin_berthelin_rouen1651"],
    "weyen" : ["cuivre_weyen_barbin_paris1669"],
    "baur" : ["cuivre_baur_sn_augsbourg1709",
          "cuivre_baur_sn_vienne1639"],
    "philippe" : ["cuivre_philippe_hackiana_leyde1670"],
    "tempesta" : ["cuivre_tempesta_dejode_anvers1606",
              "cuivre_tempesta_jansonius_amsterdam1610"],
    "borcht" : ["cuivre_borcht_plantin_anvers1591"],
    "bouche" : ["cuivre_bouche_blaeu_amsterdam1702"],
    "mathieu" : ["cuivre_mathieu_langelier_paris1619"],
    "monconet" : ["cuivre_monconet_sommaville_paris1660"],
    "ht" : ["cuivre_ht_molin_lyon1697_t4",
        "cuivre_ht_molin_lyon1697_t5",
        "cuivre_ht_molin_lyon1697_t6"],
    
    
    # À ajouter quand disponibles
    # "altzenbach" : Cöllen 1681 — Bnu Strasbourg

    # "mulder"     : Amsterdam 1683 — à sourcer            """"ressemble a philippe""""

        
}



# Vérification
print("Graveurs et illustrations disponibles :\n")
for graveur, dossiers in GRAVEURS.items():
    total = 0
    for d in dossiers:
        chemin = os.path.join(DOSSIER_SEG, d)
        if os.path.exists(chemin):
            n = len([f for f in os.listdir(chemin)
                     if f.endswith(".jpg") and "_flip" not in f])
            total += n
        else:
            print(f"  ⚠️  {d} — introuvable")
    print(f"  {graveur:15s} : {total} illustrations")

Graveurs et illustrations disponibles :

  salomon         : 164 illustrations
  solis           : 187 illustrations
  wickram         : 52 illustrations
  savery          : 18 illustrations
  de_passe        : 271 illustrations
  eskrich         : 56 illustrations
  leroy           : 44 illustrations
  gaultier        : 74 illustrations
  isaac           : 17 illustrations
  briot           : 49 illustrations
  goltzius        : 38 illustrations
  blanchin        : 18 illustrations
  weyen           : 6 illustrations
  baur            : 347 illustrations
  philippe        : 16 illustrations
  tempesta        : 309 illustrations
  borcht          : 184 illustrations
  bouche          : 129 illustrations
  mathieu         : 174 illustrations
  monconet        : 149 illustrations
  ht              : 17 illustrations


In [15]:
base_pdfs = os.path.join(RACINE, "data", "cuivre_pdfs_bruts")

print(f"Contenu de {base_pdfs} :\n")
for f in sorted(os.listdir(base_pdfs)):
    print(f"  {f}")

Contenu de /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/cuivre_pdfs_bruts :

  Les_métamorphoses_d'Ovide_traduittes_en_[...]Ovide_(0043_bpt6k722055.pdf
  Metamorphoseon_Ovidianarum_typi_aliquot_artificiosissimè_[...]Ovide_(0043_bpt6k15218623.pdf
  P_Ovid_Nasonis_XV_Metamorphoseon_[...]Salsmann_Wilhelm_bpt6k1522448r.pdf
  [Les_Métamorphoses_d'Ovide_traduites_en_[...]Ovide_(0043_bpt6k6277348n.pdf
  briot_drobet_lyon1628.pdf
  goltzius_haarlem1589.pdf


In [16]:
RENOMMAGES_PDFS = {
    "Metamorphoseon_Ovidianarum_typi_aliquot_artificiosissimè_[...]Ovide_(0043_bpt6k15218623.pdf" : "cuivre_depasse_koln1602.pdf",
    "P_Ovid_Nasonis_XV_Metamorphoseon_[...]Salsmann_Wilhelm_bpt6k1522448r.pdf"                   : "cuivre_depasse_arnhem1607.pdf",
    "[Les_Métamorphoses_d'Ovide_traduites_en_[...]Ovide_(0043_bpt6k6277348n.pdf"                 : "cuivre_gaultier_veuveguillemot1614.pdf",
    "Les_métamorphoses_d'Ovide_traduittes_en_[...]Ovide_(0043_bpt6k722055.pdf"                   : "cuivre_isaac_langelier1617.pdf",
}

base_pdfs = os.path.join(RACINE, "data", "cuivre_pdfs_bruts")

for ancien, nouveau in RENOMMAGES_PDFS.items():
    src = os.path.join(base_pdfs, ancien)
    dst = os.path.join(base_pdfs, nouveau)
    if os.path.exists(src):
        os.rename(src, dst)
        print(f"✓ {ancien[:50]}... → {nouveau}")
    else:
        print(f"  [SKIP] {ancien[:50]}... — introuvable")

✓ Metamorphoseon_Ovidianarum_typi_aliquot_artificios... → cuivre_depasse_koln1602.pdf
✓ P_Ovid_Nasonis_XV_Metamorphoseon_[...]Salsmann_Wil... → cuivre_depasse_arnhem1607.pdf
✓ [Les_Métamorphoses_d'Ovide_traduites_en_[...]Ovide... → cuivre_gaultier_veuveguillemot1614.pdf
✓ Les_métamorphoses_d'Ovide_traduittes_en_[...]Ovide... → cuivre_isaac_langelier1617.pdf


## 3. Téléchargement et segmentation — nouvelles sources

⚠️ Charger YOLO uniquement pour cette section — libérer avant le split.

Renseigner les ARKs/BSB IDs des nouvelles éditions dans les cellules ci-dessous.
Chaque graveur a sa propre cellule — facile à compléter au fur et à mesure.

In [8]:
import sys, os

RACINE = os.path.abspath("../../../")
sys.path.insert(0, RACINE)
sys.path.insert(0, os.path.join(RACINE, "yolov5_repo"))

# Vérification
print("Paths :")
for p in sys.path[:5]:
    print(f"  {p}")

# Vérifier que yolov5_repo est accessible
import importlib
spec = importlib.util.find_spec("utils")
print(f"\nutils trouvé : {spec}")
# Charger YOLO — uniquement si on a de nouvelles sources à segmenter
modele_yolo = charger_yolo(yolov5_repo=os.path.join(RACINE, "yolov5_repo"))

Paths :
  /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/yolov5_repo
  /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir
  /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/yolov5_repo
  /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir
  /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/yolov5_repo

utils trouvé : ModuleSpec(name='utils', loader=<_frozen_importlib_external.SourceFileLoader object at 0x70bc1a530ec0>, origin='/mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/yolov5_repo/utils/__init__.py', submodule_search_locations=['/mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/yolov5_repo/utils'])
✓ YOLO chargé — classes : {0: 'illustration'}


In [9]:
import requests
from io import BytesIO
import sys
sys.path.insert(0, os.path.join(RACINE, "yolov5_repo"))

BASE_URL    = "https://gallica-search-api-preprod.bnf.lajavaness.com"

# Tempesta — API BnF
r             = requests.get(f"{BASE_URL}/api/ouvrages/btv1b54000051z/illustrations", timeout=30)
illustrations = r.json()
valides       = [i for i in illustrations
                 if i.get("metas", {}).get("content_embedding")
                 and len(i["metas"]["content_embedding"]) == 768]

print(f"✓ Tempesta De Jode 1606 — {len(valides)} illustrations avec embedding")

dossier_brut = os.path.join(DOSSIER_SOURCES, "cuivre_tempesta_dejode_anvers1606")
os.makedirs(dossier_brut, exist_ok=True)
pages = []

for i, illus in enumerate(valides):
    print(f"  {i+1}/{len(valides)}...", end="\r")
    url    = illus["metas"].get("link", "")
    view   = illus.get("view_number", i)
    chemin = os.path.join(dossier_brut, f"tempesta_f{view:03d}.jpg")
    if os.path.exists(chemin):
        pages.append(chemin)
        continue
    try:
        img = Image.open(BytesIO(requests.get(url, timeout=15).content)).convert("RGB")
        img.save(chemin)
        pages.append(chemin)
    except Exception as e:
        print(f"\n  Erreur {view} : {e}")

print(f"\n✓ {len(pages)} pages téléchargées")
segmenter_corpus(pages, os.path.join(DOSSIER_SEG, "cuivre_tempesta_dejode_anvers1606"), modele_yolo, conf_thres=0.25)

# Tempesta — BSB
pages_bsb = telecharger_pages_iiif(
    "https://api.digitale-sammlungen.de/iiif/presentation/v2/bsb00008186/manifest",
    os.path.join(DOSSIER_SOURCES, "cuivre_tempesta_jansonius_amsterdam1610"),
    prefixe="tempesta"
)
segmenter_corpus(pages_bsb, os.path.join(DOSSIER_SEG, "cuivre_tempesta_jansonius_amsterdam1610"), modele_yolo, conf_thres=0.25)

✓ Tempesta De Jode 1606 — 153 illustrations avec embedding
  153/153...
✓ 153 pages téléchargées
  153/153...
✓ 155 illustrations extraites dans /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/segmentees/cuivre_tempesta_dejode_anvers1606
Pages trouvées : 306 — Tempesta, Antonio: Metamorphoseon Sive Transformationvm Ovid
  306/306...
✓ 306 pages sauvegardées dans /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/sources/cuivre_tempesta_jansonius_amsterdam1610
  306/306...
✓ 157 illustrations extraites dans /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/segmentees/cuivre_tempesta_jansonius_amsterdam1610


157

In [10]:
pages_borcht = telecharger_pages_iiif(
    "https://api.digitale-sammlungen.de/iiif/presentation/v2/bsb00004340/manifest",
    os.path.join(DOSSIER_SOURCES, "cuivre_borcht_plantin_anvers1591"),
    prefixe="borcht"
)
segmenter_corpus(
    pages_borcht,
    os.path.join(DOSSIER_SEG, "cuivre_borcht_plantin_anvers1591"),
    modele_yolo,
    conf_thres=0.25
)

Pages trouvées : 386 — P. Ovidii Nasonis Metamorphoses, Argumentis breuioribus, ex 
  386/386...
✓ 386 pages sauvegardées dans /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/sources/cuivre_borcht_plantin_anvers1591
  386/386...
✓ 184 illustrations extraites dans /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/segmentees/cuivre_borcht_plantin_anvers1591


184

In [15]:
import fitz

RACINE      = os.path.abspath("../../../")
DOSSIER_PDFS    = os.path.join(RACINE, "data", "cuivre_pdfs_bruts")
DOSSIER_SOURCES = os.path.join(RACINE, "data", "sources")
DOSSIER_SEG     = os.path.join(RACINE, "data", "segmentees")

def extraire_pages_pdf(chemin_pdf, dossier_sortie, dpi=150):
    os.makedirs(dossier_sortie, exist_ok=True)
    doc   = fitz.open(chemin_pdf)
    pages = []
    for i, page in enumerate(doc):
        mat    = fitz.Matrix(dpi/72, dpi/72)
        pix    = page.get_pixmap(matrix=mat)
        chemin = f"{dossier_sortie}/page{i+1:03d}.jpg"
        pix.save(chemin)
        pages.append(chemin)
        print(f"  {i+1}/{len(doc)}...", end="\r")
    print(f"\n✓ {len(pages)} pages extraites depuis {os.path.basename(chemin_pdf)}")
    return pages

PDFS_BOUCHE = {
    "cuivre_bouche_blaeu_amsterdam1702.pdf": "cuivre_bouche_blaeu_amsterdam1702"
}

for fichier, nom in PDFS_BOUCHE.items():
    chemin_pdf = os.path.join(DOSSIER_PDFS, fichier)
    if not os.path.exists(chemin_pdf):
        print(f"⚠️  {fichier} introuvable")
        continue
    print(f"\nTraitement : {fichier}")
    pages = extraire_pages_pdf(chemin_pdf, os.path.join(DOSSIER_SOURCES, nom))
    segmenter_corpus(pages, os.path.join(DOSSIER_SEG, nom), modele_yolo, conf_thres=0.25)


Traitement : cuivre_bouche_blaeu_amsterdam1702.pdf
  603/603...
✓ 603 pages extraites depuis cuivre_bouche_blaeu_amsterdam1702.pdf
  603/603...
✓ 129 illustrations extraites dans /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/segmentees/cuivre_bouche_blaeu_amsterdam1702


In [16]:
for ark, label in [
    ("btv1b22000826", "Mathieu — L'Angelier Paris 1619"),
    ("bpt6k87019014", "Mathieu — Billaine Paris 1637"),
]:
    r = requests.get(f"{BASE_URL}/api/ouvrages/{ark}/illustrations", timeout=15)
    data = r.json()
    if isinstance(data, list):
        print(f"✓ {label} : {len(data)} illustrations — API BnF")
    else:
        r2 = requests.get(f"https://gallica.bnf.fr/iiif/ark:/12148/{ark}/manifest.json", timeout=15)
        print(f"✗ {label} — API : {data.get('message','')} | IIIF : HTTP {r2.status_code}")


✓ Mathieu — L'Angelier Paris 1619 : 195 illustrations — API BnF
✗ Mathieu — Billaine Paris 1637 — API : Ouvrage bpt6k87019014 not found. | IIIF : HTTP 403


In [17]:
r             = requests.get(f"{BASE_URL}/api/ouvrages/btv1b22000826/illustrations", timeout=30)
illustrations = r.json()
valides       = [i for i in illustrations
                 if i.get("metas", {}).get("content_embedding")
                 and len(i["metas"]["content_embedding"]) == 768]

print(f"✓ Mathieu — {len(valides)} illustrations avec embedding")

dossier_brut = os.path.join(DOSSIER_SOURCES, "cuivre_mathieu_langelier_paris1619")
os.makedirs(dossier_brut, exist_ok=True)
pages = []

for i, illus in enumerate(valides):
    print(f"  {i+1}/{len(valides)}...", end="\r")
    url    = illus["metas"].get("link", "")
    view   = illus.get("view_number", i)
    chemin = os.path.join(dossier_brut, f"mathieu_f{view:03d}.jpg")
    if os.path.exists(chemin):
        pages.append(chemin)
        continue
    try:
        img = Image.open(BytesIO(requests.get(url, timeout=15).content)).convert("RGB")
        img.save(chemin)
        pages.append(chemin)
    except Exception as e:
        print(f"\n  Erreur {view} : {e}")

print(f"\n✓ {len(pages)} pages téléchargées")
segmenter_corpus(pages, os.path.join(DOSSIER_SEG, "cuivre_mathieu_langelier_paris1619"), modele_yolo, conf_thres=0.25)

✓ Mathieu — 195 illustrations avec embedding
  195/195...
✓ 195 pages téléchargées
  195/195...
✓ 196 illustrations extraites dans /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/segmentees/cuivre_mathieu_langelier_paris1619


196

In [20]:
PDFS_MONCONET = {
    "cuivre_monconet_sommaville_paris1660.pdf": "cuivre_monconet_sommaville_paris1660"
}

for fichier, nom in PDFS_MONCONET.items():
    chemin_pdf = os.path.join(DOSSIER_PDFS, fichier)
    if not os.path.exists(chemin_pdf):
        print(f"⚠️  {fichier} introuvable")
        continue
    print(f"\nTraitement : {fichier}")
    pages = extraire_pages_pdf(chemin_pdf, os.path.join(DOSSIER_SOURCES, nom))
    segmenter_corpus(pages, os.path.join(DOSSIER_SEG, nom), modele_yolo, conf_thres=0.25)


Traitement : cuivre_monconet_sommaville_paris1660.pdf
  764/764...
✓ 764 pages extraites depuis cuivre_monconet_sommaville_paris1660.pdf
  764/764...
✓ 149 illustrations extraites dans /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/segmentees/cuivre_monconet_sommaville_paris1660


### 3.1 Eskrich, Pierre
##### <u>ARK / BSB :</u> à renseigner
##### <u>Source :</u> à renseigner
##### <u>Dossier :</u> `bois_eskrich_{ville}{année}/`

In [7]:
import requests, os
from PIL import Image
from io import BytesIO

BASE_URL    = "https://gallica-search-api-preprod.bnf.lajavaness.com"

ARKS_ESKRICH = {
    "btv1b22000559" : "eskrich_rouille1556",
    "bpt6k8709995p" : "eskrich_bonhomme1556",
}

for ark, nom in ARKS_ESKRICH.items():
    r             = requests.get(f"{BASE_URL}/api/ouvrages/{ark}/illustrations", timeout=30)
    illustrations = r.json()
    valides       = [i for i in illustrations
                     if i.get("metas", {}).get("content_embedding")
                     and len(i["metas"]["content_embedding"]) == 768]

    print(f"\n{nom} — {len(valides)} illustrations avec embedding")

    dossier_brut = os.path.join(DOSSIER_SOURCES, f"bois_{nom}")
    os.makedirs(dossier_brut, exist_ok=True)
    pages = []

    for i, illus in enumerate(valides):
        print(f"  {i+1}/{len(valides)}...", end="\r")
        url    = illus["metas"].get("link", "")
        view   = illus.get("view_number", i)
        chemin = os.path.join(dossier_brut, f"{nom}_f{view:03d}.jpg")
        if os.path.exists(chemin):
            pages.append(chemin)
            continue
        try:
            img = Image.open(BytesIO(requests.get(url, timeout=15).content)).convert("RGB")
            img.save(chemin)
            pages.append(chemin)
        except Exception as e:
            print(f"\n  Erreur {view} : {e}")

    print(f"\n✓ {len(pages)} pages téléchargées")

    # Segmenter
    segmenter_corpus(
        pages,
        os.path.join(DOSSIER_SEG, f"bois_{nom}"),
        modele_yolo,
        conf_thres=0.25
    )


eskrich_rouille1556 — 66 illustrations avec embedding
  66/66...
✓ 66 pages téléchargées
  66/66...
✓ 61 illustrations extraites dans /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/segmentees/bois_eskrich_rouille1556


AttributeError: 'str' object has no attribute 'get'

## Biblioteca Digital Ovidiana (ovidiuspictus.es)

### 3.2 Leroy II, Guillaume
##### <u>ARK / BSB :</u> à renseigner
##### <u>Source :</u> à renseigner
##### <u>Dossier :</u> `bois_leroy_{ville}{année}/`

In [11]:
ARKS_LEROY = {
    "bsb11054210" : "leroy_gueynard1510",
}

for bsb_id, nom in ARKS_LEROY.items():
    pages = telecharger_pages_iiif(
        f"https://api.digitale-sammlungen.de/iiif/presentation/v2/{bsb_id}/manifest",
        os.path.join(DOSSIER_SOURCES, f"bois_{nom}"),
        prefixe=nom
    )
    segmenter_corpus(
        pages,
        os.path.join(DOSSIER_SEG, f"bois_{nom}"),
        modele_yolo,
        conf_thres=0.25
    )

Pages trouvées : 504 — P. Ouidij Nasonis metamorphoseos libri moralizati: cum pulch
  504/504...
✓ 504 pages sauvegardées dans /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/sources/bois_leroy_gueynard1510
  504/504...
✓ 44 illustrations extraites dans /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/segmentees/bois_leroy_gueynard1510


## Biblioteca Digital Ovidiana (ovidiuspictus.es)

Source complémentaire utilisée pour récupérer des illustrations non accessibles
via l'API BnF ou BSB Munich.

La **Biblioteca Digital Ovidiana** est un projet de recherche espagnol qui recense
et numérise les éditions illustrées des *Métamorphoses* d'Ovide du 15e au 18e siècle.
Chaque édition indexée dispose d'une page dédiée listant ses illustrations sous forme
de vignettes téléchargeables.

**Avantage :** les illustrations sont déjà découpées et présentées individuellement —
pas besoin de segmentation YOLO.

**Limite :** certaines pages incluent des frontispices ou illustrations de titre
qui ne correspondent pas à des scènes des fables ovidiennes — à exclure manuellement.

**Accès :** `http://www.ovidiuspictus.es` — images téléchargeables via web scraping
depuis les pages `ilustracionesejemplar.php?clave={id}`.

In [21]:
import requests, os
from bs4 import BeautifulSoup
from PIL import Image
from io import BytesIO

BASE_OVIDIUS = "http://www.ovidiuspictus.es"
DOSSIER_SEG  = os.path.join(os.path.abspath("../../../"), "data", "segmentees")

def telecharger_depuis_clave(clave, nom_dossier):
    """
    Télécharge toutes les illustrations d'un exemplaire depuis Biblioteca Digital Ovidiana.
    Paramètres :
        clave       : int ou str — le clave de ilustracionesejemplar
        nom_dossier : str — nom du dossier de destination dans data/segmentees/
    """
    url  = f"{BASE_OVIDIUS}/en/ilustracionesejemplar.php?clave={clave}"
    r    = requests.get(url, timeout=15)
    soup = BeautifulSoup(r.text, "html.parser")
    imgs = [img.get("src", "") for img in soup.find_all("img")
            if "/images/images/" in img.get("src", "")]
    dossier = os.path.join(DOSSIER_SEG, nom_dossier)
    os.makedirs(dossier, exist_ok=True)
    print(f"\n{nom_dossier} — clave={clave} — {len(imgs)} illustrations")
    for i, src in enumerate(imgs):
        print(f"  {i+1}/{len(imgs)}...", end="\r")
        url_img     = f"{BASE_OVIDIUS}/{src.replace('../', '')}"
        nom_fichier = f"ex{clave}_{os.path.basename(src)}"
        chemin      = os.path.join(dossier, nom_fichier)
        if os.path.exists(chemin):
            continue
        try:
            r_img = requests.get(url_img, timeout=15)
            img   = Image.open(BytesIO(r_img.content)).convert("RGB")
            img.save(chemin)
        except Exception as e:
            print(f"\n  Erreur {nom_fichier} : {e}")
    n = len([f for f in os.listdir(dossier) if f.endswith(".jpg")])
    print(f"\n✓ {n} illustrations sauvegardées → {nom_dossier}")
    return n

# monogrammiste H.T. — Molin, Lyon 1697
telecharger_depuis_clave(52, "cuivre_ht_molin_lyon1697_t4")
telecharger_depuis_clave(53, "cuivre_ht_molin_lyon1697_t5")
telecharger_depuis_clave(54, "cuivre_ht_molin_lyon1697_t6")


cuivre_ht_molin_lyon1697_t4 — clave=52 — 6 illustrations
  6/6...
✓ 6 illustrations sauvegardées → cuivre_ht_molin_lyon1697_t4

cuivre_ht_molin_lyon1697_t5 — clave=53 — 6 illustrations
  6/6...
✓ 6 illustrations sauvegardées → cuivre_ht_molin_lyon1697_t5

cuivre_ht_molin_lyon1697_t6 — clave=54 — 5 illustrations
  5/5...
✓ 5 illustrations sauvegardées → cuivre_ht_molin_lyon1697_t6


5

### 3.3 Gaultier, Léonard
##### <u>ARK / BSB :</u> à renseigner
##### <u>Source :</u> à renseigner
##### <u>Dossier :</u> `cuivre_gaultier_{ville}{année}/`

In [12]:
SOURCES_GAULTIER = {
    "bsb11913284" : "cuivre_gaultier_guillemot1610",
    "bsb10242228" : "cuivre_gaultier_1616",
}

for bsb_id, nom in SOURCES_GAULTIER.items():
    pages = telecharger_pages_iiif(
        f"https://api.digitale-sammlungen.de/iiif/presentation/v2/{bsb_id}/manifest",
        os.path.join(DOSSIER_SOURCES, nom),
        prefixe=nom
    )
    segmenter_corpus(
        pages,
        os.path.join(DOSSIER_SEG, nom),
        modele_yolo,
        conf_thres=0.25
    )

Pages trouvées : 846 — Ovidius Naso, Publius: Les Metamorphoses d'Ovide
  846/846...
✓ 846 pages sauvegardées dans /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/sources/cuivre_gaultier_guillemot1610
  846/846...
✓ 27 illustrations extraites dans /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/segmentees/cuivre_gaultier_guillemot1610
Pages trouvées : 1242 — Ovidius Naso, Publius: Les quinze livres de la metamorphose 
  1242/1242...
✓ 1242 pages sauvegardées dans /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/sources/cuivre_gaultier_1616
  1242/1242...
✓ 31 illustrations extraites dans /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/segmentees/cuivre_gaultier_1616


## Biblioteca Digital Ovidiana (ovidiuspictus.es)

### 3.4 Isaac, Jaspar
##### <u>ARK / BSB :</u> à renseigner
##### <u>Source :</u> à renseigner
##### <u>Dossier :</u> `cuivre_isaac_{ville}{année}/`

## Biblioteca Digital Ovidiana (ovidiuspictus.es)

### 3.5 Briot, Isaac
##### <u>ARK / BSB :</u> à renseigner
##### <u>Source :</u> à renseigner
##### <u>Dossier :</u> `cuivre_briot_{ville}{année}/`

In [14]:
import fitz

DOSSIER_PDFS = os.path.join(RACINE, "data", "cuivre_pdfs_bruts")

PDFS_BRIOT = {
    "briot_drobet_lyon1628.pdf": "cuivre_briot_drobet1628"
}

def extraire_pages_pdf(chemin_pdf, dossier_sortie, dpi=150):
    """Convertit chaque page d'un PDF en JPG."""
    os.makedirs(dossier_sortie, exist_ok=True)
    doc   = fitz.open(chemin_pdf)
    pages = []
    for i, page in enumerate(doc):
        mat    = fitz.Matrix(dpi/72, dpi/72)
        pix    = page.get_pixmap(matrix=mat)
        chemin = f"{dossier_sortie}/page{i+1:03d}.jpg"
        pix.save(chemin)
        pages.append(chemin)
        print(f"  {i+1}/{len(doc)}...", end="\r")
    print(f"\n✓ {len(pages)} pages extraites depuis {os.path.basename(chemin_pdf)}")
    return pages

for fichier, nom in PDFS_BRIOT.items():
    chemin_pdf = os.path.join(DOSSIER_PDFS, fichier)
    if not os.path.exists(chemin_pdf):
        print(f"⚠️  {fichier} introuvable dans {DOSSIER_PDFS}")
        continue
    print(f"\nTraitement : {fichier}")
    pages = extraire_pages_pdf(chemin_pdf, os.path.join(DOSSIER_SOURCES, nom))
    segmenter_corpus(pages, os.path.join(DOSSIER_SEG, nom), modele_yolo, conf_thres=0.25)


Traitement : briot_drobet_lyon1628.pdf
  748/748...
✓ 748 pages extraites depuis briot_drobet_lyon1628.pdf
  748/748...
✓ 49 illustrations extraites dans /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/segmentees/cuivre_briot_drobet1628


## Biblioteca Digital Ovidiana (ovidiuspictus.es)

### 3.6 Baur, Johann Wilhelm
##### <u>ARK / BSB :</u> à renseigner
##### <u>Source :</u> à renseigner
##### <u>Dossier :</u> `cuivre_baur_{ville}{année}/`

In [18]:
SOURCES_BAUR = {
    "bsb10872075" : "cuivre_baur_sn_augsbourg1709",
    "bsb10872073" : "cuivre_baur_sn_vienne1639",
}

for bsb_id, nom in SOURCES_BAUR.items():
    pages = telecharger_pages_iiif(
        f"https://api.digitale-sammlungen.de/iiif/presentation/v2/{bsb_id}/manifest",
        os.path.join(DOSSIER_SOURCES, nom),
        prefixe=nom
    )
    segmenter_corpus(
        pages,
        os.path.join(DOSSIER_SEG, nom),
        modele_yolo,
        conf_thres=0.25
    )

Pages trouvées : 458 — Baur, Johann Wilhelm: Des vortrefflichen römischen Poetens P
  458/458...
✓ 458 pages sauvegardées dans /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/sources/cuivre_baur_sn_augsbourg1709


NameError: name 'modele_yolo' is not defined

In [23]:
import sys
RACINE = os.path.abspath("../../../")
sys.path.insert(0, RACINE)
sys.path.insert(0, os.path.join(RACINE, "yolov5_repo"))

for bsb_id, nom in SOURCES_BAUR.items():
    dossier_source = os.path.join(DOSSIER_SOURCES, nom)
    
    # Charger les pages déjà téléchargées si le dossier existe
    if os.path.exists(dossier_source) and len(os.listdir(dossier_source)) > 0:
        pages = sorted([
            os.path.join(dossier_source, f)
            for f in os.listdir(dossier_source)
            if f.endswith(".jpg")
        ])
        print(f"✓ {nom} — {len(pages)} pages déjà disponibles")
    else:
        pages = telecharger_pages_iiif(
            f"https://api.digitale-sammlungen.de/iiif/presentation/v2/{bsb_id}/manifest",
            dossier_source,
            prefixe=nom
        )

    segmenter_corpus(
        pages,
        os.path.join(DOSSIER_SEG, nom),
        modele_yolo,
        conf_thres=0.25
    )

✓ cuivre_baur_sn_augsbourg1709 — 458 pages déjà disponibles
  458/458...
✓ 172 illustrations extraites dans /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/segmentees/cuivre_baur_sn_augsbourg1709
Pages trouvées : 246 — Baur, Johann Wilhelm: Ovid's Verwandlungen
  246/246...
✓ 246 pages sauvegardées dans /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/sources/cuivre_baur_sn_vienne1639
  246/246...
✓ 175 illustrations extraites dans /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/segmentees/cuivre_baur_sn_vienne1639


## Biblioteca Digital Ovidiana (ovidiuspictus.es)

### 3.7 Altzenbach, Gerhardt
##### <u>ARK / BSB :</u> à renseigner
##### <u>Source :</u> à renseigner
##### <u>Dossier :</u> `cuivre_altzenbach_{ville}{année}/`

In [ ]:
A numériser

## Biblioteca Digital Ovidiana (ovidiuspictus.es)

### 3.8 Goltzius, Hendrick
##### <u>ARK / BSB :</u> à renseigner
##### <u>Source :</u> à renseigner
##### <u>Dossier :</u> `cuivre_goltzius_{ville}{année}/`

In [15]:
PDFS_GOLTZIUS = {
    "goltzius_haarlem1589.pdf": "cuivre_goltzius_haarlem1589"
}

for fichier, nom in PDFS_GOLTZIUS.items():
    chemin_pdf = os.path.join(DOSSIER_PDFS, fichier)
    if not os.path.exists(chemin_pdf):
        print(f"⚠️  {fichier} introuvable dans {DOSSIER_PDFS}")
        continue
    print(f"\nTraitement : {fichier}")
    pages = extraire_pages_pdf(chemin_pdf, os.path.join(DOSSIER_SOURCES, nom))
    segmenter_corpus(pages, os.path.join(DOSSIER_SEG, nom), modele_yolo, conf_thres=0.25)


Traitement : goltzius_haarlem1589.pdf
  65/65...
✓ 65 pages extraites depuis goltzius_haarlem1589.pdf
  65/65...
✓ 38 illustrations extraites dans /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/segmentees/cuivre_goltzius_haarlem1589


In [16]:
# Conversion en niveaux de gris — illustrations Goltzius (colorées)
from PIL import Image
import os

dossier_couleur = os.path.join(DOSSIER_SEG, "cuivre_goltzius_haarlem1589_couleur")
dossier_gris    = os.path.join(DOSSIER_SEG, "cuivre_goltzius_haarlem1589")

# Renommer l'ancien dossier en _couleur
if os.path.exists(dossier_gris) and not os.path.exists(dossier_couleur):
    os.rename(dossier_gris, dossier_couleur)
    print(f"✓ Renommé en : cuivre_goltzius_haarlem1589_couleur")

# Créer le nouveau dossier en niveaux de gris
os.makedirs(dossier_gris, exist_ok=True)

images = [f for f in os.listdir(dossier_couleur) if f.endswith(".jpg")]

for nom in images:
    img = Image.open(os.path.join(dossier_couleur, nom)).convert("L").convert("RGB")
    img.save(os.path.join(dossier_gris, nom))
    print(f"  {nom}", end="\r")

print(f"\n✓ {len(images)} illustrations converties en niveaux de gris → cuivre_goltzius_haarlem1589")

✓ Renommé en : cuivre_goltzius_haarlem1589_couleur
  page054_det1_conf0.72.jpg
✓ 38 illustrations converties en niveaux de gris → cuivre_goltzius_haarlem1589


## Biblioteca Digital Ovidiana (ovidiuspictus.es)

### 3.9 Mulder, Joseph
##### <u>ARK / BSB :</u> à renseigner
##### <u>Source :</u> à renseigner
##### <u>Dossier :</u> `cuivre_mulder_{ville}{année}/`

In [ ]:
Rien

## Biblioteca Digital Ovidiana (ovidiuspictus.es)

### 3.10 Weyen, Laurent
##### <u>ARK / BSB :</u> à renseigner
##### <u>Source :</u> à renseigner
##### <u>Dossier :</u> `cuivre_weyen_{ville}{année}/`

In [ ]:
Rien

## Biblioteca Digital Ovidiana (ovidiuspictus.es)

### 3.11 Blanchin, Jean
##### <u>ARK / BSB :</u> à renseigner
##### <u>Source :</u> à renseigner
##### <u>Dossier :</u> `cuivre_blanchin_{ville}{année}/`

In [ ]:
Rien

## Biblioteca Digital Ovidiana (ovidiuspictus.es)

### 3.12 Philippe, Pierre
##### <u>ARK / BSB :</u> à renseigner
##### <u>Source :</u> à renseigner
##### <u>Dossier :</u> `cuivre_philippe_{ville}{année}/`

In [ ]:
Rien

ValueError: too many values to unpack (expected 2)

## Biblioteca Digital Ovidiana (ovidiuspictus.es)

## 4. Libérer la mémoire GPU

⚠️ Obligatoire avant le split.

In [17]:
modele_yolo = liberer_yolo(modele_yolo)

NameError: name 'modele_yolo' is not defined

## 5. Récapitulatif des illustrations disponibles

In [22]:
print("Illustrations disponibles par graveur :\n")
print(f"  {'Graveur':20s} {'Dossier':45s} {'Nb':>5}")
print("  " + "─" * 75)

for graveur, dossiers in GRAVEURS.items():
    total = 0
    for d in dossiers:
        chemin = os.path.join(DOSSIER_SEG, d)
        if os.path.exists(chemin):
            n = len([f for f in os.listdir(chemin)
                     if f.endswith(".jpg") and "_flip" not in f])
            print(f"  {graveur:20s} {d:45s} {n:>5}")
            total += n
        else:
            print(f"  {graveur:20s} {d:45s} {'N/A':>5}")
    if len(dossiers) > 1:
        print(f"  {'':20s} {'total':45s} {total:>5}")
    print()

Illustrations disponibles par graveur :

  Graveur              Dossier                                          Nb
  ───────────────────────────────────────────────────────────────────────────
  salomon              bois_salomon_lyon1557                           164

  solis                bois_solis_francfort1581                        187

  wickram              bois_wickram_mayence1545                         52

  savery               cuivre_clein_paris1637                           18

  de_passe             cuivre_depasse_koln1602                         135
  de_passe             cuivre_depasse_arnhem1607                       136
                       total                                           271

  eskrich              bois_eskrich_rouille1556                         56

  leroy                bois_leroy_gueynard1510                          44

  gaultier             cuivre_gaultier_guillemot1610                    27
  gaultier             cuivre_gaultier_1616      

## 6. Rassembler les illustrations par graveur

Copie toutes les illustrations (originaux + flips) dans `data/datasets/graveur/{graveur}/`.

In [ ]:
def rassembler_par_graveur(graveurs, dossier_seg, dossier_dataset):
    """
    Copie toutes les illustrations de chaque graveur
    dans data/datasets/graveur/{graveur}/
    """
    for graveur, dossiers in graveurs.items():
        dest = os.path.join(dossier_dataset, graveur)
        os.makedirs(dest, exist_ok=True)

        nb = 0
        for d in dossiers:
            chemin = os.path.join(dossier_seg, d)
            if not os.path.exists(chemin):
                print(f"  ⚠️  {d} introuvable")
                continue
            for f in os.listdir(chemin):
                if f.endswith(".jpg"):
                    shutil.copy(
                        os.path.join(chemin, f),
                        os.path.join(dest, f"{d}_{f}")
                    )
                    nb += 1
        print(f"  {graveur:20s} : {nb} images copiées → {dest}")

rassembler_par_graveur(GRAVEURS, DOSSIER_SEG, DOSSIER_DATASET)
print("\n✓ Rassemblement terminé")

## 7. Flip horizontal — Data augmentation

Génère les versions flippées dans des dossiers `{graveur}_flip/`.

In [ ]:
def flipper_graveur(graveur, dossier_dataset):
    dossier      = os.path.join(dossier_dataset, graveur)
    dossier_flip = os.path.join(dossier_dataset, f"{graveur}_flip")
    os.makedirs(dossier_flip, exist_ok=True)

    images = [f for f in os.listdir(dossier)
              if f.endswith(".jpg") and "_flip" not in f]
    nb = 0
    for nom in images:
        dest = os.path.join(dossier_flip, nom)
        if not os.path.exists(dest):
            img = Image.open(os.path.join(dossier, nom)).convert("RGB")
            ImageOps.mirror(img).save(dest)
            nb += 1
    return nb

print("Data augmentation — flip horizontal :\n")
for graveur in GRAVEURS:
    nb = flipper_graveur(graveur, DOSSIER_DATASET)
    print(f"  {graveur:20s} : +{nb} flips")
print("\n✓ Augmentation terminée")

## 8. Split train / val / test

Split 70/15/15 — flips uniquement dans train.

In [ ]:
def split_graveur(dossier_dataset, graveurs, ratio_train=0.7, ratio_val=0.15):
    for split in ["train", "val", "test"]:
        for graveur in graveurs:
            os.makedirs(os.path.join(dossier_dataset, split, graveur), exist_ok=True)

    print("Split train/val/test :\n")

    for graveur in graveurs:
        dossier      = os.path.join(dossier_dataset, graveur)
        dossier_flip = os.path.join(dossier_dataset, f"{graveur}_flip")

        originaux = [f for f in os.listdir(dossier) if f.endswith(".jpg")]
        random.shuffle(originaux)

        n       = len(originaux)
        n_train = int(n * ratio_train)
        n_val   = int(n * ratio_val)

        splits = {
            "train": originaux[:n_train],
            "val"  : originaux[n_train:n_train + n_val],
            "test" : originaux[n_train + n_val:]
        }

        comptes = {"train": 0, "val": 0, "test": 0}

        for split, fichiers in splits.items():
            for f in fichiers:
                shutil.copy(
                    os.path.join(dossier, f),
                    os.path.join(dossier_dataset, split, graveur, f)
                )
                comptes[split] += 1

                # Flip uniquement dans train
                if split == "train" and os.path.exists(dossier_flip):
                    chemin_flip = os.path.join(dossier_flip, f)
                    if os.path.exists(chemin_flip):
                        shutil.copy(
                            chemin_flip,
                            os.path.join(dossier_dataset, "train", graveur, f"flip_{f}")
                        )
                        comptes["train"] += 1

        print(f"  {graveur:20s} — train:{comptes['train']:>4} | val:{comptes['val']:>4} | test:{comptes['test']:>4}")

# Nettoyer les splits existants
for split in ["train", "val", "test"]:
    chemin = os.path.join(DOSSIER_DATASET, split)
    if os.path.exists(chemin):
        shutil.rmtree(chemin)

split_graveur(DOSSIER_DATASET, list(GRAVEURS.keys()))

print("\n✓ Dataset final :")
for split in ["train", "val", "test"]:
    total = sum(
        len(os.listdir(os.path.join(DOSSIER_DATASET, split, g)))
        for g in GRAVEURS
        if os.path.exists(os.path.join(DOSSIER_DATASET, split, g))
    )
    print(f"  {split:5s} : {total} images")

In [24]:
print("Récapitulatif final du dataset graveur :\n")
print(f"  {'Graveur':15s} {'Illustrations':>15}   {'Dossier(s)'}")
print("  " + "─" * 75)

total_global = 0
for graveur, dossiers in GRAVEURS.items():
    total_graveur = 0
    dossiers_info = []
    for d in dossiers:
        chemin = os.path.join(DOSSIER_SEG, d)
        if os.path.exists(chemin):
            n = len([f for f in os.listdir(chemin)
                     if f.endswith(".jpg") and "_flip" not in f])
            total_graveur += n
            dossiers_info.append(f"{d} ({n})")
        else:
            dossiers_info.append(f"{d} (N/A)")
    print(f"  {graveur:15s} {total_graveur:>15}   {', '.join(dossiers_info)}")
    total_global += total_graveur

print("  " + "─" * 75)
print(f"  {'TOTAL':15s} {total_global:>15}   {len(GRAVEURS)} graveurs")

Récapitulatif final du dataset graveur :

  Graveur           Illustrations   Dossier(s)
  ───────────────────────────────────────────────────────────────────────────
  salomon                     164   bois_salomon_rouille_lyon1557 (164)
  solis                       187   bois_solis_feyerabend_francfort1581 (187)
  wickram                      52   bois_wickram_behem_mayence1545 (52)
  savery                       18   cuivre_savery_farnaby_paris1637 (18)
  de_passe                    271   cuivre_depasse_depasse_koln1602 (135), cuivre_depasse_jansonius_arnhem1607 (136)
  eskrich                      56   bois_eskrich_rouille_lyon1556 (56)
  leroy                        44   bois_leroy_gueynard_lyon1510 (44)
  gaultier                     74   cuivre_gaultier_guillemot_paris1610 (27), cuivre_gaultier_sn_paris1616 (31), cuivre_gaultier_veuveguillemot_paris1614 (16)
  isaac                        17   cuivre_isaac_langelier_paris1617 (17)
  briot                        49   cuivre_brio